# Method 2 — Transformer-Based Code Classifiers (CodeBERT, GraphCodeBERT, CodeGPT)

**Fixes Reviewer #3, M5:** these are **encoder/decoder-based transformer
classifiers**, not generative LLMs used unsupervised. Naming reflects that
throughout.

**Model lineup and why:**

| Model | Architecture | Why it's here |
|---|---|---|
| **CodeBERT** (`microsoft/codebert-base`) | RoBERTa encoder | Pretrained on 6 languages incl. Python (CodeSearchNet) |
| **GraphCodeBERT** (`microsoft/graphcodebert-base`) | RoBERTa encoder + data-flow pretraining | Same corpus as CodeBERT, adds structural signal |
| **CodeGPT** (`microsoft/CodeGPT-small-py-adaptedGPT2`) | GPT-2 **decoder-only** | Genuinely different architecture family (decoder-only vs. encoder-only), pretrained specifically on Python |

**Note on CodeT5 / VulBERTa / LineVul:** VulBERTa and LineVul were found to
be C/C++-specific (see earlier notes) and were replaced with GraphCodeBERT
and CodeT5 originally. CodeT5 was then dropped and replaced with CodeGPT
after extensive debugging revealed the installed `transformers` build in
this environment has multiple internal bugs specifically affecting T5's
architecture (`find_pruneable_heads_and_indices` import, then a
`_tied_weights_keys`-as-list bug affecting several model families
including T5) that are open, unresolved upstream issues, not something
fixable from notebook code. CodeGPT (GPT-2 architecture) sidesteps this
entirely while still providing a genuinely different architecture family
(decoder-only) from CodeBERT/GraphCodeBERT (encoder-only).

**Two real bugs fixed from the original version:**
- `DEVICE` was used but never defined -- would crash with `NameError`.
- The `find_pruneable_heads_and_indices` monkey-patch below is kept as a
  defensive measure in case T5-family models are reintroduced later once
  the upstream bug is fixed.

**If you hit `ImportError`/`AttributeError` chains from `transformers`
itself (not from this notebook's code) when running this:** that indicates
a broken/inconsistent library install, not a code bug. Fix with:
```
!pip uninstall -y transformers tokenizers
!pip install --force-reinstall --no-cache-dir transformers
```
then **restart the runtime** before re-running.

**Outputs saved (per model):**
- `models/<model_key>_finetuned/` -- fine-tuned weights + tokenizer
- `models/<model_key>_scores.csv` -- per-sample probabilities for every split (id, split, prob)
- `results/llm_metrics.json` -- accuracy/precision/recall/F1/AUC for all models, keyed by name
- `results/llm_roc_comparison.png` -- combined ROC/AUC curve across all models
- `results/llm_confusion_matrices.png` -- confusion matrices side by side


In [ ]:
!pip install torch transformers accelerate scikit-learn pandas numpy matplotlib seaborn tqdm --quiet


In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────
import os, json, warnings, random
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from datetime import datetime
from typing import List

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                              confusion_matrix, classification_report, roc_auc_score,
                              roc_curve)

warnings.filterwarnings('ignore')
random.seed(42); np.random.seed(42); torch.manual_seed(42)

os.makedirs('models', exist_ok=True)
os.makedirs('results', exist_ok=True)

# THE FIX: DEVICE was used but never defined in the previous version.
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
print(f'Timestamp: {datetime.now().strftime("%Y-%m-%d %H:%M")}')


In [ ]:
# ── Defensive patch: restores a function some transformers 5.x builds are
# missing for T5-family models. Harmless no-op for CodeBERT/GraphCodeBERT/
# CodeGPT; kept in case a T5-family model is added back later. ────────────
import transformers.pytorch_utils as _ptu

if not hasattr(_ptu, 'find_pruneable_heads_and_indices'):
    def find_pruneable_heads_and_indices(heads, n_heads, head_size, already_pruned_heads):
        mask = torch.ones(n_heads, head_size)
        heads = set(heads) - already_pruned_heads
        for head in heads:
            head = head - sum(1 if h < head else 0 for h in already_pruned_heads)
            mask[head] = 0
        mask = mask.view(-1).contiguous().eq(1)
        index = torch.arange(len(mask))[mask].long()
        return heads, index
    _ptu.find_pruneable_heads_and_indices = find_pruneable_heads_and_indices
    print('Patched: find_pruneable_heads_and_indices')
else:
    print('find_pruneable_heads_and_indices already present -- no patch needed')


In [ ]:
# ── Load dataset ──────────────────────────────────────────────────────────
CSV = 'dataset_with_content.csv'
if not os.path.exists(CSV):
    raise FileNotFoundError('Run the dataset-collection notebook first to generate dataset_with_content.csv')

df = pd.read_csv(CSV)
df.columns = df.columns.str.strip().str.lower()
df = df.dropna(subset=['content', 'label']).reset_index(drop=True)
df['label'] = df['label'].astype(int)
if 'id' not in df.columns:
    df['id'] = df.index

print(f'Dataset: {len(df):,} samples')
print(f'Vulnerable: {(df["label"]==1).sum():,}   Safe: {(df["label"]==0).sum():,}')


## Models and hyperparameters

Same fine-tuning recipe (epochs, learning rate) applied identically to all
models. `BATCH_SIZE` defaults to 8 for CPU-only safety; raise to 16 on GPU
(Colab T4 etc.) for faster training.


In [ ]:
# ── Models to compare ─────────────────────────────────────────────────────
MODELS = [
    {'key': 'codebert',      'name': 'CodeBERT',      'hf_id': 'microsoft/codebert-base'},
    {'key': 'graphcodebert', 'name': 'GraphCodeBERT',  'hf_id': 'microsoft/graphcodebert-base'},
    {'key': 'codegpt',       'name': 'CodeGPT',        'hf_id': 'microsoft/CodeGPT-small-py-adaptedGPT2'},
]

MAX_LEN    = 256
BATCH_SIZE = 8    # raise to 16 on GPU (Colab T4 etc.); keep at 8 or lower on CPU-only
EPOCHS     = 3
LR         = 2e-5
WARMUP     = 0.1

print('Models to fine-tune:')
for m in MODELS:
    print(f"  - {m['name']} ({m['hf_id']})")
print(f'\nBATCH_SIZE={BATCH_SIZE}, MAX_LEN={MAX_LEN}')


In [ ]:
# ── PyTorch Dataset ───────────────────────────────────────────────────────
class CodeDataset(Dataset):
    def __init__(self, codes: List[str], labels: List[int], tokenizer, max_len: int = MAX_LEN):
        self.codes, self.labels = codes, labels
        self.tokenizer, self.max_len = tokenizer, max_len

    def __len__(self):
        return len(self.codes)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.codes[idx], truncation=True, max_length=self.max_len,
            padding='max_length', return_tensors='pt'
        )
        return {
            'input_ids': enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'label': torch.tensor(self.labels[idx], dtype=torch.long),
        }


In [ ]:
# ── Train / Val / Test split (same seed as every other notebook) ─────────
idx_all = np.arange(len(df))
y_all = df['label'].values

idx_tv, idx_test = train_test_split(idx_all, test_size=0.20, random_state=42, stratify=y_all)
idx_train, idx_val = train_test_split(idx_tv, test_size=0.25, random_state=42, stratify=y_all[idx_tv])

print(f'Train: {len(idx_train):,}   Val: {len(idx_val):,}   Test: {len(idx_test):,}')


## Fine-tune each model in turn

Progress is logged with elapsed time so you can see exactly where things
stand. Each finished model is checkpointed to disk -- if a later model
crashes or the runtime disconnects, re-running this cell skips every
already-finished model instead of retraining from scratch.


In [ ]:
# ── Fine-tune + evaluate every model ─────────────────────────────────────
import sys, time as _time

def _log(msg):
    """Prints immediately with a timestamp, bypassing any output buffering
    so progress is visible even in environments/execution modes that don't
    flush stdout live by default (e.g. some remote or scripted runs)."""
    elapsed = _time.time() - _RUN_START
    print(f'[{elapsed/60:6.1f} min] {msg}', flush=True)
    sys.stdout.flush()

_RUN_START = _time.time()
all_results = {}   # name -> dict with metrics, probs, preds, trues

_log(f'Starting fine-tuning run: {len(MODELS)} model(s) queued -> '
     f'{", ".join(m["name"] for m in MODELS)}')

SKIP_ALREADY_DONE = True   # set False to force re-training every model from scratch

for m_idx, m in enumerate(MODELS, 1):
    name, hf_id, key = m['name'], m['hf_id'], m['key']
    is_gpt_style = 'gpt' in hf_id.lower()
    print(f'\n{"="*70}\n[{m_idx}/{len(MODELS)}] {name} ({hf_id})\n{"="*70}', flush=True)

    # ── Checkpoint resume: if this model already finished in a previous run
    #    (e.g. before a crash on a LATER model), load its saved results
    #    instead of re-training it from scratch. ───────────────────────────
    scores_path = f'models/{key}_scores.csv'
    model_dir = f'models/{key}_finetuned'
    if SKIP_ALREADY_DONE and os.path.exists(scores_path) and os.path.isdir(model_dir):
        _log(f'{name}: found existing results at {scores_path} -- skipping training '
             f'(checkpoint resume). Set SKIP_ALREADY_DONE=False above to force redo.')
        _scores_existing = pd.read_csv(scores_path)
        _test_scores = _scores_existing[_scores_existing['split'] == 'test']
        _merged = _test_scores.merge(df[['id', 'label']], on='id', how='inner')
        test_probs = _merged[f'{key}_prob'].values
        test_trues = _merged['label'].values
        test_preds = (test_probs >= 0.5).astype(int)
        acc = accuracy_score(test_trues, test_preds)
        prec = precision_score(test_trues, test_preds, zero_division=0)
        rec = recall_score(test_trues, test_preds, zero_division=0)
        f1 = f1_score(test_trues, test_preds, zero_division=0)
        auc_val = roc_auc_score(test_trues, test_probs)
        cm = confusion_matrix(test_trues, test_preds)
        _log(f'{name} loaded from checkpoint: Acc={acc:.4f}  F1={f1:.4f}  AUC={auc_val:.4f}')
        all_results[name] = {
            'key': key, 'accuracy': acc, 'precision': prec, 'recall': rec, 'f1': f1, 'auc': auc_val,
            'confusion_matrix': cm, 'test_probs': test_probs, 'test_trues': test_trues,
            'train_losses': [], 'val_accs': [], 'val_f1s': [],
        }
        continue

    _log(f'Downloading/loading tokenizer for {name} ({hf_id}) -- first run for a '
         f'model can take a while if it needs to download from Hugging Face...')
    tokenizer = AutoTokenizer.from_pretrained(hf_id)
    if is_gpt_style and tokenizer.pad_token is None:
        # GPT-2 style tokenizers have no pad token by default
        tokenizer.pad_token = tokenizer.eos_token
    _log(f'Tokenizer ready for {name}.')

    train_ds = CodeDataset(df['content'].iloc[idx_train].tolist(), y_all[idx_train].tolist(), tokenizer)
    val_ds   = CodeDataset(df['content'].iloc[idx_val].tolist(),   y_all[idx_val].tolist(),   tokenizer)
    test_ds  = CodeDataset(df['content'].iloc[idx_test].tolist(),  y_all[idx_test].tolist(),  tokenizer)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

    _log(f'Downloading/loading pretrained weights for {name} -- this is the slow step '
         f'on first run (can be several hundred MB to ~1 GB depending on the model).')
    model = AutoModelForSequenceClassification.from_pretrained(hf_id, num_labels=2).to(DEVICE)
    _log(f'{name} model loaded on {DEVICE}. Starting training...')

    # GPT-2 style models need pad_token_id wired into the model config so the
    # classification head knows where each sequence actually ends.
    if is_gpt_style:
        model.config.pad_token_id = tokenizer.pad_token_id

    # T5-family models need these for the classification head's EOS-pooling
    # and internal decoder-input shifting (kept defensively; no-op otherwise).
    if getattr(model.config, 'decoder_start_token_id', None) is None:
        model.config.decoder_start_token_id = tokenizer.pad_token_id
    if getattr(model.config, 'eos_token_id', None) is None and tokenizer.eos_token_id is not None:
        model.config.eos_token_id = tokenizer.eos_token_id

    optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    total_steps = len(train_loader) * EPOCHS
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=int(total_steps * WARMUP), num_training_steps=total_steps)
    loss_fn = nn.CrossEntropyLoss()

    train_losses, val_accs, val_f1s = [], [], []
    for epoch in range(EPOCHS):
        model.train()
        epoch_loss = 0.0
        for batch in tqdm(train_loader, desc=f'{name} epoch {epoch+1}/{EPOCHS} train'):
            ids  = batch['input_ids'].to(DEVICE)
            mask = batch['attention_mask'].to(DEVICE)
            lbls = batch['label'].to(DEVICE)
            optimizer.zero_grad()
            out = model(input_ids=ids, attention_mask=mask)
            loss = loss_fn(out.logits, lbls)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            epoch_loss += loss.item()
        avg_loss = epoch_loss / len(train_loader)
        train_losses.append(avg_loss)
        _log(f'{name} epoch {epoch+1}/{EPOCHS} training pass complete (avg loss={avg_loss:.4f}), '
             f'now running validation...')

        model.eval()
        vp, vt = [], []
        with torch.no_grad():
            for batch in val_loader:
                ids  = batch['input_ids'].to(DEVICE)
                mask = batch['attention_mask'].to(DEVICE)
                lbls = batch['label'].numpy()
                out = model(input_ids=ids, attention_mask=mask)
                vp.extend(out.logits.argmax(dim=1).cpu().numpy())
                vt.extend(lbls)
        vacc, vf1 = accuracy_score(vt, vp), f1_score(vt, vp, zero_division=0)
        val_accs.append(vacc); val_f1s.append(vf1)
        _log(f'{name} epoch {epoch+1}/{EPOCHS} DONE: loss={avg_loss:.4f}  '
             f'val_acc={vacc:.4f}  val_f1={vf1:.4f}')

    _log(f'{name} training complete. Saving model + scoring test set...')
    os.makedirs(f'models/{key}_finetuned', exist_ok=True)
    model.save_pretrained(f'models/{key}_finetuned')
    tokenizer.save_pretrained(f'models/{key}_finetuned')

    # ── Score every split (for Hybrid.ipynb / app.py, same pattern as Bandit/RL) ──
    def score_loader(loader):
        model.eval()
        probs_out, preds_out, trues_out = [], [], []
        with torch.no_grad():
            for batch in loader:
                ids  = batch['input_ids'].to(DEVICE)
                mask = batch['attention_mask'].to(DEVICE)
                lbls = batch['label'].numpy()
                out = model(input_ids=ids, attention_mask=mask)
                prbs = torch.softmax(out.logits, dim=1).cpu().numpy()
                probs_out.extend(prbs[:, 1])
                preds_out.extend(out.logits.argmax(dim=1).cpu().numpy())
                trues_out.extend(lbls)
        return np.array(probs_out), np.array(preds_out), np.array(trues_out)

    train_probs, _, _ = score_loader(train_loader)
    val_probs, _, _ = score_loader(val_loader)
    test_probs, test_preds, test_trues = score_loader(test_loader)

    scores_df = pd.concat([
        pd.DataFrame({'id': df['id'].iloc[idx_train].values, 'split': 'train', f'{key}_prob': train_probs}),
        pd.DataFrame({'id': df['id'].iloc[idx_val].values,   'split': 'val',   f'{key}_prob': val_probs}),
        pd.DataFrame({'id': df['id'].iloc[idx_test].values,  'split': 'test',  f'{key}_prob': test_probs}),
    ], ignore_index=True)
    scores_df.to_csv(f'models/{key}_scores.csv', index=False)

    acc = accuracy_score(test_trues, test_preds)
    prec = precision_score(test_trues, test_preds, zero_division=0)
    rec = recall_score(test_trues, test_preds, zero_division=0)
    f1 = f1_score(test_trues, test_preds, zero_division=0)
    auc_val = roc_auc_score(test_trues, test_probs)
    cm = confusion_matrix(test_trues, test_preds)

    print(f'\n=== {name} — Test Set ===')
    print(f'Accuracy={acc:.4f}  Precision={prec:.4f}  Recall={rec:.4f}  F1={f1:.4f}  AUC={auc_val:.4f}')

    all_results[name] = {
        'key': key, 'accuracy': acc, 'precision': prec, 'recall': rec, 'f1': f1, 'auc': auc_val,
        'confusion_matrix': cm, 'test_probs': test_probs, 'test_trues': test_trues,
        'train_losses': train_losses, 'val_accs': val_accs, 'val_f1s': val_f1s,
    }

    _log(f'[{m_idx}/{len(MODELS)}] {name} fully done -- '
         f'Acc={acc:.4f} F1={f1:.4f} AUC={auc_val:.4f}')
    # Explicit memory cleanup between models -- important on CPU-only, low-RAM
    # setups where three transformer models loaded in sequence without this
    # can accumulate enough memory pressure to crash the kernel.
    del model, tokenizer, train_loader, val_loader, test_loader, train_ds, val_ds, test_ds
    import gc
    gc.collect()
    torch.cuda.empty_cache() if torch.cuda.is_available() else None
    _log(f'Memory cleaned up after {name}.')

_log('All models fine-tuned and evaluated.')
print('\nAll models fine-tuned and evaluated.')


## Metrics comparison table


In [ ]:
# ── Metrics comparison table ──────────────────────────────────────────────
comparison_df = pd.DataFrame([
    {'Model': name, 'Accuracy': r['accuracy'], 'Precision': r['precision'],
     'Recall': r['recall'], 'F1': r['f1'], 'AUC': r['auc']}
    for name, r in all_results.items()
]).set_index('Model')

print(comparison_df.round(4).to_string())


## Combined ROC/AUC comparison

Direct visual comparison of all evaluated transformer models on the same
test set -- a curve further toward the top-left with higher AUC
discriminates vulnerable from safe code better across every threshold.


In [ ]:
# ── Combined ROC curve + confusion matrices ──────────────────────────────
BG, AXB, CT, CG = '#0f172a', '#1e293b', '#f1f5f9', '#334155'
MODEL_COLORS = {'CodeBERT': '#3b82f6', 'GraphCodeBERT': '#22c55e', 'CodeGPT': '#f59e0b',
                 'CodeT5': '#a855f7'}

fig, ax = plt.subplots(figsize=(7, 7), facecolor=BG)
ax.set_facecolor(AXB)
for name, r in all_results.items():
    fpr, tpr, _ = roc_curve(r['test_trues'], r['test_probs'])
    ax.plot(fpr, tpr, linewidth=2.5, color=MODEL_COLORS.get(name),
            label=f"{name} (AUC = {r['auc']:.4f})")
ax.plot([0, 1], [0, 1], linestyle='--', color=CG, label='Random guess (AUC = 0.50)')
ax.set_xlabel('False Positive Rate', color=CT)
ax.set_ylabel('True Positive Rate', color=CT)
ax.set_title('Transformer Model Comparison — ROC Curves', color=CT, fontweight='bold')
ax.tick_params(colors=CT)
ax.legend(loc='lower right', facecolor=AXB, labelcolor=CT)
ax.grid(alpha=0.2, color=CG)
plt.tight_layout()
plt.savefig('results/llm_roc_comparison.png', dpi=300, facecolor=BG, bbox_inches='tight')
plt.show()
print('Saved: results/llm_roc_comparison.png')

fig, axes = plt.subplots(1, len(all_results), figsize=(6*len(all_results), 5), facecolor=BG)
if len(all_results) == 1:
    axes = [axes]
for ax, (name, r) in zip(axes, all_results.items()):
    ax.set_facecolor(AXB)
    sns.heatmap(r['confusion_matrix'], annot=True, fmt='d', ax=ax, cmap='RdYlGn',
                xticklabels=['Safe', 'Vulnerable'], yticklabels=['Safe', 'Vulnerable'],
                cbar=False, annot_kws={'size': 14, 'color': 'white'})
    ax.set_title(name, color=CT, fontweight='bold')
    ax.set_xlabel('Predicted', color='#94a3b8')
    ax.set_ylabel('Actual', color='#94a3b8')
    ax.tick_params(colors='#94a3b8')
plt.tight_layout()
plt.savefig('results/llm_confusion_matrices.png', dpi=300, facecolor=BG, bbox_inches='tight')
plt.show()
print('Saved: results/llm_confusion_matrices.png')


In [ ]:
# ── Save metrics ────────────────────────────────────────────────────────
metrics_out = {}
for name, r in all_results.items():
    tn, fp, fn, tp = r['confusion_matrix'].ravel()
    metrics_out[name] = {
        'hf_id': next(m['hf_id'] for m in MODELS if m['name'] == name),
        'accuracy': round(float(r['accuracy']), 4),
        'precision': round(float(r['precision']), 4),
        'recall': round(float(r['recall']), 4),
        'f1': round(float(r['f1']), 4),
        'auc': round(float(r['auc']), 4),
        'confusion_matrix': {'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp)},
        'training': {'epochs': EPOCHS, 'batch_size': BATCH_SIZE, 'max_len': MAX_LEN, 'lr': LR,
                     'train_losses': r['train_losses'], 'val_accs': r['val_accs'], 'val_f1s': r['val_f1s']},
    }

with open('results/llm_metrics.json', 'w') as f:
    json.dump(metrics_out, f, indent=2)

print('Saved: results/llm_metrics.json')
print(json.dumps({k: {kk: vv for kk, vv in v.items() if kk != 'training'} for k, v in metrics_out.items()}, indent=2))


## Next steps

1. Update Section IV-B / II-B of the paper: "Transformer-based semantic
   module" comparing CodeBERT, GraphCodeBERT, and CodeGPT (Encoder-only x2,
   Decoder-only x1 -- a genuine cross-architecture comparison).
2. `Hybrid.ipynb` should pick the best-performing model here (by AUC) as
   the transformer component, loading `models/<key>_scores.csv`.
3. `app.py`'s `METHOD_REGISTRY` and `inference_utils.py`'s
   `_KNOWN_TRANSFORMER_KEYS` should include an entry for `codegpt` (and
   drop `codet5` unless it gets reintroduced later).
4. If CodeT5 is reintroduced once the upstream `transformers` T5 bugs are
   fixed, re-add it to `MODELS` and use `RobertaTokenizer` explicitly for
   it (per Salesforce's own model card) rather than `AutoTokenizer`.
